# From an Existing Notebook to a Deployable Pipeline
### Instructor demonstration — Bank Loan Example

**Learning objective:** Keep the original notebook workflow, then add a *deployment preparation* section that combines preprocessing and prediction into one saved object.

This notebook uses a **small synthetic dataset** for teaching. Predictions are illustrative and must not be used for real lending decisions.

**Run cells from top to bottom.** In Google Colab, the required packages are normally available.

## Part A — What students have ALREADY done

Imagine the student has completed data preparation, train/test splitting, scaling, training and evaluation. We reproduce that process below so you can compare it with the new pipeline.

**Teaching point:** Students do not delete these cells or redo their project.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import joblib

rng = np.random.default_rng(42)
n = 160
income = rng.integers(3500, 26000, size=n)
loan = rng.integers(10000, 150000, size=n)
credit_score = rng.integers(450, 851, size=n)
# Synthetic outcome for illustration only; not a real lending policy.
score = (income / 5000) - (loan / 60000) + (credit_score - 600) / 110
approved = (score + rng.normal(0, 0.55, size=n) > 2.0).astype(int)
df = pd.DataFrame({"monthly_income": income, "loan_amount": loan,
                   "credit_score": credit_score, "approved": approved})
print(df.head())
print("Rows:", len(df))

   monthly_income  loan_amount  credit_score  approved
0            5508       149454           578         0
1           20914       103003           493         0
2           18227        67286           696         1
3           13374        66894           817         1
4           13242        68514           841         1
Rows: 160


In [3]:
X = df[["monthly_income", "loan_amount", "credit_score"]]
y = df["approved"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))

Training rows: 120 Test rows: 40


### Original approach: preprocessing and model are separate

The scaler learns its statistics **only from training data**. The test data is transformed using the already-fitted scaler.

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
old_predictions = model.predict(X_test_scaled)
print("Original model test accuracy:", round(accuracy_score(y_test, old_predictions), 3))

Original model test accuracy: 0.85


## Part B — NEW section: Preparing the Model for Deployment

This is the section students add **after their original project**. We use the same original training columns, before scaling, and the same train/test split.

**What does `Pipeline` do?** It remembers the sequence: (1) transform inputs using the fitted scaler, (2) send transformed inputs to the fitted model. `fit()` fits both in order. `predict()` applies the saved transformations and then predicts.

In [5]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

# IMPORTANT: X_train is UNSCALED here. The pipeline handles scaling.
pipeline.fit(X_train, y_train)
print(pipeline)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])


### Check the new pipeline before deployment

Do not report the old model's performance as the pipeline's performance without checking. Evaluate the pipeline on the held-out test set.

In [6]:
pipeline_predictions = pipeline.predict(X_test)
print("Pipeline test accuracy:", round(accuracy_score(y_test, pipeline_predictions), 3))
print("Same test predictions as the original approach?",
      np.array_equal(old_predictions, pipeline_predictions))

Pipeline test accuracy: 0.85
Same test predictions as the original approach? True


### Test with a new business input

Notice that we provide **raw values** (not manually scaled). Column names and data types must match training. The pipeline automatically performs scaling.

In [7]:
new_customer = pd.DataFrame([{
    "monthly_income": 12000,
    "loan_amount": 60000,
    "credit_score": 720
}])

prediction = pipeline.predict(new_customer)[0]
print("Demonstration prediction:", "Approved" if prediction == 1 else "Not approved")

Demonstration prediction: Approved


### Save the whole pipeline in one file

`pipeline.pkl` contains both the fitted scaler and fitted classifier. Only load `.pkl` files you created yourself or obtained from a trusted source.

In [8]:
joblib.dump(pipeline, "pipeline.pkl")
print("Saved pipeline.pkl")

Saved pipeline.pkl


### Simulate Streamlit loading the saved pipeline

This is the important deployment test: load the file into a new variable and predict without manually scaling.

In [9]:
loaded_pipeline = joblib.load("pipeline.pkl")
loaded_prediction = loaded_pipeline.predict(new_customer)[0]
print("Prediction from saved pipeline:", "Approved" if loaded_prediction == 1 else "Not approved")

Prediction from saved pipeline: Approved


## Part C — Optional: Minimal Streamlit code

Copy the code below into a separate file named `app.py` **when ready to deploy**. This cell writes the file for you. The `app.py` and `pipeline.pkl` files should be in the same GitHub repository, alongside `requirements.txt`.

In a real student project, use the actual features and model, not this demonstration data.

In [10]:
app_code = 'import streamlit as st\nimport pandas as pd\nimport joblib\n\nst.title("Bank Loan Prediction — Teaching Demo")\nst.caption("Synthetic educational model; not for real lending decisions.")\n\npipeline = joblib.load("pipeline.pkl")\nincome = st.number_input("Monthly income (AED)", min_value=0, value=12000)\nloan = st.number_input("Loan amount (AED)", min_value=0, value=60000)\ncredit = st.number_input("Credit score", min_value=300, max_value=850, value=720)\n\nif st.button("Predict"):\n    customer = pd.DataFrame([{\n        "monthly_income": income,\n        "loan_amount": loan,\n        "credit_score": credit\n    }])\n    prediction = pipeline.predict(customer)[0]\n    st.write("Demo prediction:", "Approved" if prediction == 1 else "Not approved")\n'
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("streamlit\npandas\nscikit-learn\njoblib\n")
print("Created app.py and requirements.txt")

Created app.py and requirements.txt


## Instructor discussion questions

1. Why do we pass **unscaled** `X_train` to `pipeline.fit()`?
2. What happens automatically when `pipeline.predict(new_customer)` runs?
3. Why is it important to evaluate the final pipeline, even if the original model was already evaluated?
4. Which three files do we need for a simple Streamlit deployment?

**Answers:** (1) Scaling is already inside the pipeline. (2) It scales using training statistics, then predicts. (3) The pipeline is a newly fitted artifact and must be verified. (4) `app.py`, `pipeline.pkl`, `requirements.txt`.

**Adaptation note:** If a student uses missing-value imputation, categorical encoding, feature engineering or feature selection, those steps must also be included in the deployment process. A simple scaler-only pipeline is not enough for every project. Never fit preprocessing on the test set or on a single new customer.